In [ ]:
import heapq
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [ ]:
N_PEERS = 130
N_TARGET = 100
T_BEGIN = 69
T_END = 170
N_ITERS = 5
RESULTS_PATH = "results"

In [ ]:
def parse_file(filepath):
    """Parse a file into a list of (int, str, str) tuples."""
    result = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            result.append((int(parts[0]), parts[1], parts[2]))
    return result

In [ ]:
def merge_sorted_files(file_data_list):
    """
    Merge N sorted arrays into one sorted array using a min-heap.
    """
    # Min-heap: (int_value, file_index, entry_index, str1, str2)
    heap = []
    
    # Initialize heap with first element from each non-empty file
    for j in range(len(file_data_list)):
        entry = file_data_list[j][0]
        # Push: (int_value, j, entry_index, str1, str2)
        heapq.heappush(heap, (entry[0], j, 0, entry[1], entry[2]))
    
    result = []
    while heap:
        int_val, j, idx, str1, str2 = heapq.heappop(heap)
        # Append with source as second element: (int, "h{j}", str, str)
        result.append((int_val, f"h{j+1}", str1, str2))
        
        # If there are more entries in this file, push the next one
        next_idx = idx + 1
        if next_idx < len(file_data_list[j]):
            next_entry = file_data_list[j][next_idx]
            heapq.heappush(heap, (next_entry[0], j, next_idx, next_entry[1], next_entry[2]))
    
    return result

In [ ]:
def read_seed(n, seed):
    file_data_list = []
    for i in range(n):
        file_name = f"../{RESULTS_PATH}/{n}/{seed}/evnt_h{i+1}.log"
        file_data = parse_file(file_name)
        if len(file_data) > 0:
            file_data_list.append(file_data)
        else:
            print(f"  [warn] read_seed: file '{file_name}' is empty")

    return merge_sorted_files(file_data_list)

In [ ]:
class NDGraph:
    def __init__(self):
        self.hosts = {}
        self.links = {}

    def process(self, event):
        _, host, etype, uuid = event

        if etype == 'E':
            self.hosts[host] = uuid
            self.links[uuid] = set()

        elif etype == 'J':
            my_uuid = self.hosts[host]
            if my_uuid == uuid:
                raise Exception("self join")
            self.links[my_uuid].add(uuid)

        elif etype == 'L':
            my_uuid = self.hosts[host]
            self.links[my_uuid].remove(uuid)

        elif etype == 'X':
            my_uuid = self.hosts.pop(host)
            self.links.pop(my_uuid)

    def stats(self):
        link_count = 0
        for links in self.links.values():
            link_count += len(links)

        host_count = len(self.hosts)
        expected_count = host_count * (host_count - 1)
    
        return link_count, expected_count

In [ ]:
def write_file(path, n, seed):
    data = read_seed(n, seed)
    with open(path, "w") as f:
        graph = NDGraph()
        for event in tqdm(data):
            graph.process(event)
            link_count, link_count_expected = graph.stats()
            if link_count_expected == 0:
                f.write(f'{event[0]} N/A\n')
            else:
                f.write(f'{event[0]} {link_count/(N_TARGET*(N_TARGET-1))}\n')
        link_count, _ = graph.stats()
        return link_count/(N_TARGET*(N_TARGET-1))
        

In [ ]:
def main():
    with open(f'../{RESULTS_PATH}/{N_PEERS}/r_term.txt', "w") as f:
        for i in range(N_ITERS):
            terminal_reachability = write_file(f'../{RESULTS_PATH}/{N_PEERS}/r_{i}.txt', N_PEERS, i)
            f.write(f'{terminal_reachability}\n')

    for i in range(N_ITERS):
        with open(f'../{RESULTS_PATH}/{N_PEERS}/r_{i}.txt', 'r') as f:
            time_init = 0
            time_points = []
            reachabilities = []
            
            for line in f:
                parts = line.strip().split()
                if parts[1] == 'N/A':
                    continue

                time = int(parts[0])
                if time_init == 0:
                    time_init = time
                execution_time = (time - time_init) / 1000
                reachability = float(parts[1])
                    
                # time interpolation
                if len(time_points) > 0:
                    time_points.append(execution_time)
                    reachabilities.append(reachabilities[-1])
                
                time_points.append(execution_time)
                reachabilities.append(reachability * 100)
            if time_points[-1] < T_END:
                time_points.append(float(T_END))
                reachabilities.append(reachabilities[-1])
                
            plt.plot(time_points, reachabilities, color='blue', alpha=0.3)

    plt.ylim(80, 101)
    plt.xlim(T_BEGIN, T_END)
    plt.xlabel('Time (s)')
    plt.ylabel('Reachability (%)')
    plt.grid(True)
    plt.savefig(f'../{RESULTS_PATH}/{N_PEERS}_r_plot.jpg', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
for path in [f'continuous/{pkg}' for pkg in ['client-server', 'dev-eval-trickle', 'dev-v2']]:
    for n_peers in [110, 130, 200]:
        N_PEERS = n_peers
        RESULTS_PATH = path
        main()